# Pima Indians Diabetes 데이터 기초 EDA

결측치, 중복값, 이상치를 확인하고 시각화합니다.

## 1. 데이터 불러오기

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

path = kagglehub.dataset_download("kumargh/pimaindiansdiabetescsv")
print("Path to dataset files:", path)

csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
csv_path = os.path.join(path, csv_file)

columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

df = pd.read_csv(csv_path, header=None, names=columns)

print(df.shape)
df.head()

## 2. 결측치 확인

- 진짜 빈 값(NaN)이 있는지 확인
- 이 데이터셋은 결측치를 `0`으로 표기해놓은 경우가 많아 (예: 혈당 0은 있을 수 없음) 그 부분도 함께 확인

In [ ]:
print("=== NaN 개수 ===")
print(df.isnull().sum())

plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Value Heatmap")
plt.show()

In [ ]:
zero_as_missing_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("=== 0 값 개수 (실질적 결측치로 의심되는 값) ===")
zero_counts = (df[zero_as_missing_cols] == 0).sum()
print(zero_counts)

zero_counts.plot(kind="bar", figsize=(8, 5), color="orange")
plt.title("Count of Zero Values (Suspected Missing Data)")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

## 3. 중복값 확인

In [ ]:
dup_count = df.duplicated().sum()
print(f"중복된 행 개수: {dup_count}")

if dup_count > 0:
    print(df[df.duplicated()])

## 4. 이상치 확인 (박스플롯 + IQR)

In [ ]:
feature_cols = [c for c in df.columns if c != "Outcome"]

plt.figure(figsize=(15, 8))
for i, col in enumerate(feature_cols):
    plt.subplot(2, 4, i + 1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
print("=== IQR 기준 이상치 개수 ===")
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)}개 (범위: {lower:.2f} ~ {upper:.2f})")